In [387]:
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from utils import vprint

In [160]:
def print_metadata(tcga_project:str=None, tcga_dir:str='./../../datasets/tcga/', return_df:bool=False):
    # read df
    df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_metadata.csv')

    # drop 'Unnamed: 0' col if applicable
    if 'Unnamed: 0' in df.columns:
            df = df.drop(columns='Unnamed: 0')

    # for each metadata column
    for i in df.columns:
        # get unique items per column
        unique = df[i].unique().tolist()

        # print items per column
        if len(unique) > 15:
            print(f'{i} ({len(unique)}): {unique[0:3]+['...']}\n') # shorten if > 15 unique
        else:
            print(f'{i} ({len(unique)}): {unique}\n')

    if return_df == True:
        return df
        

In [161]:
print_metadata(tcga_project='TCGA-BRCA')

barcode (1231): ['TCGA-A2-A25D-01A-12R-A16F-07', 'TCGA-BH-A201-01A-11R-A14M-07', 'TCGA-AC-A23C-01A-12R-A169-07', '...']

patient (1095): ['TCGA-A2-A25D', 'TCGA-BH-A201', 'TCGA-AC-A23C', '...']

sample (1226): ['TCGA-A2-A25D-01A', 'TCGA-BH-A201-01A', 'TCGA-AC-A23C-01A', '...']

shortLetterCode (3): ['TP', 'NT', 'TM']

definition (3): ['Primary solid Tumor', 'Solid Tissue Normal', 'Metastatic']

sample_submitter_id (1226): ['TCGA-A2-A25D-01A', 'TCGA-BH-A201-01A', 'TCGA-AC-A23C-01A', '...']

sample_type_id (3): [1, 11, 6]

tumor_descriptor (3): ['Primary', 'Not Applicable', 'Metastatic']

sample_id (1226): ['eda0ff5c-84cf-44cb-888f-0aab17441dfa', '7893162b-3d19-4c54-86ee-aa4884d84e8a', '9cbd4bd6-244b-4274-a230-a7bd02e79571', '...']

submitter_id (1095): ['TCGA-A2-A25D', 'TCGA-BH-A201', 'TCGA-AC-A23C', '...']

sample_type (3): ['Primary Tumor', 'Solid Tissue Normal', 'Metastatic']

oct_embedded (2): [True, False]

specimen_type (2): ['Solid Tissue', 'Unknown']

state (1): ['released']

is_

In [ ]:
# class Metadata():
#     def __init__(self, tcga_project:str, subtype_col:str, tcga_dir:str='./../../datasets/tcga/'):
#         # read
#         self.metadata_complete = self._read_metadata(tcga_project, tcga_dir)

#         # get summary
#         self.summary = self.metadata_complete[['barcode', 'name', 'tissue_type', 'sample_type', subtype_col]]

#         # get type / subtype
#         self.metadata = self._format_metadata(self.summary, subtype_col)

#     def _read_metadata(self, tcga_project:str, tcga_dir:str):
#         # read df
#         df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_metadata.csv')

#         # drop 'Unnamed: 0' col if applicable
#         if 'Unnamed: 0' in df.columns:
#                 df = df.drop(columns='Unnamed: 0')

#         return df
    
#     def _format_metadata(self, summary:pd.DataFrame, subtype_col:str):
#         # compile df
#         df = pd.DataFrame(
#             {
#                 'barcode': summary['barcode'],
#                 'type': summary.apply(lambda row: row['name'] if row['tissue_type'] == 'Tumor' else row['tissue_type'], axis=1),
#                 'subtype': summary[subtype_col].fillna(summary['sample_type'])
#             }
#         )

#         return df
    
# metadata = Metadata(tcga_project='TCGA-BRCA', subtype_col='paper_BRCA_Subtype_PAM50')

---

In [ ]:
class Data():
    def __init__(
            self,
            # dataset 
            tcga_project:str,
            tcga_dir:str='./../../datasets/tcga/',
            relation_filepath:str='./../../datasets/other/relation_ohe.csv',

            # metadata
            metadata_subtype_col:str='',
            metadata_ctrl:str='Normal',

            # gene_counts
            log0_method:str='log1p',
            scale_gene_counts:bool=True, 
            fc_gene_counts:bool=True, 
            log2fc_gene_counts:bool=True,

            # filter, resample, etc.
            y_col:str='type',
            y_drop:list[str]=None,
            max_subset:int=None,
            verbose:bool=True,
        ):
        
        vprint('**** Data() ****', verbose=verbose)

        # assign
        self.log0_method = log0_method
        
        # read files into df
        gene_counts = self._read_gene_counts(tcga_project, tcga_dir)
        metadata = self._read_metadata(tcga_project, tcga_dir, metadata_subtype_col)
        relation = pd.read_csv(relation_filepath)

        # scale gene_counts
        if scale_gene_counts == True:
            gene_counts = self._scale_gene_counts(gene_counts)

        if (fc_gene_counts == True) & (metadata_ctrl != ''):
            gene_counts = self._fc_gene_counts(gene_counts, metadata, metadata_ctrl)

            if log2fc_gene_counts == True:
                gene_counts = np.log2(self._handle_log0(gene_counts))

        # preprocess gene_counts and relation into graph data
        gene_counts, relation, node_id_map = self._graph_preprocessing(gene_counts, relation)

        # get masks
        masks = self._get_masks(relation)

        # filter counts by class (drop classes, downsample) if applicable
        gene_counts, metadata = self._filter_counts(gene_counts, metadata, y_col, y_drop, max_subset)

        # get xy
        X, y, y_labels = self._get_Xy(gene_counts, metadata, y_col)

        # get dims
        num_samples, num_nodes, num_features, num_labels, num_masks = self._get_dims(X, y, masks)
            
        # assign
        self.gene_counts = gene_counts
        self.metadata = metadata
        self.relation = relation
        self.node_id_map = node_id_map
        self.masks = masks
        self.X = X
        self.y = y
        self.y_labels = y_labels

        self.num_samples = num_samples
        self.num_nodes = num_nodes
        self.num_features = num_features
        self.num_labels = num_labels
        self.num_masks = num_masks

        # print
        vprint(self, verbose=verbose)
        
    def _read_metadata(self, tcga_project:str, tcga_dir:str, metadata_subtype_col:str):
        # read df
        df_complete = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_metadata.csv')

        # drop 'Unnamed: 0' col if applicable
        if 'Unnamed: 0' in df_complete.columns:
                df_complete = df_complete.drop(columns='Unnamed: 0')

        # compile df
        df = pd.DataFrame(
            {
                'barcode': df_complete['barcode'],
                'type': df_complete.apply(lambda row: row['name'] if row['tissue_type'] == 'Tumor' else row['tissue_type'], axis=1),
            }
        )

        # append subtype if applicable
        if metadata_subtype_col != '':
            df['subtype'] = df_complete[metadata_subtype_col].fillna(df_complete['sample_type'])

        return df

    def _read_gene_counts(self, tcga_project:str, tcga_dir:str):
        # read df
        df = pd.read_csv(Path(tcga_dir) / f'{tcga_project}_gene_counts.csv')

        # rename 'Unnamed: 0' col if applicable
        if 'Unnamed: 0' in df.columns:
            df = df.rename(columns={'Unnamed: 0':'ensg'})

        # remove ensg version
        df['ensg'] = df['ensg'].str.split('.').str[0]

        # set index, column name
        df = df.set_index('ensg')
        df.columns.name = 'barcode'

        return df
    
    def _handle_log0(self, x):
        # offset (replace 0 with small number)
        if self.log0_method == 'offset':
            if type(x) == pd.DataFrame:
                x = x.replace(0, 1e-6)
            else:
                x = x + 1e-6 # if x is not a df, add 1e-6

        # log1p method (add 1 before log)
        elif self.log0_method == 'log1p':
            x = x + 1

        # error msg
        else:
            print("Invalid log0 method; log0 method not applied. Use 'offset' or 'log1p'.")

        return x

    def _scale_gene_counts(self, gene_counts:pd.DataFrame):
        # handle log zero
        gene_counts = self._handle_log0(gene_counts)

        # take the (natural) log of all the values
        log_counts = np.log(gene_counts)

        # average each row (e.g., geometric average)
        geom_avg = log_counts.mean(axis=1)

        # filter out +-inf
        geom_avg_filt = geom_avg[(abs(geom_avg) != np.inf)]

        # subtract the average log value from the log(counts)
        log_ratio = log_counts.sub(geom_avg_filt, axis=0)

        # caclulate the median of the ratios for each sample
        log_ratio_median = log_ratio.median()

        # calculate scaling factors (e^log_ratio_median)
        scaling_factor = log_ratio_median.apply(lambda x: np.exp(x))

        # divide the original read counts by the scaling factors; return output
        return gene_counts.div(scaling_factor, axis=1)
    
    def _fc_gene_counts(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, metadata_ctrl:str):
        # get control barcodes
        ctrl_barcodes = metadata[metadata['type'] == metadata_ctrl]['barcode']

        # get mean
        ctrl_avg = gene_counts[ctrl_barcodes].mean(axis=1).values.reshape(-1, 1) # reshape to allow division

        # get fc
        return gene_counts / ctrl_avg

    def _graph_preprocessing(self, gene_counts:pd.DataFrame, relation:pd.DataFrame):
        # filter gene_counts by relation ensembl
        unique_ensg = pd.concat([relation[cols] for cols in ['ensembl1', 'ensembl2']]).unique().tolist()
        gene_counts = gene_counts.loc[unique_ensg,:]

        # get id maps
        node_id_map = {node: int(i) for i, node in enumerate(gene_counts.index)}

        # map relation ensembl id to nodes
        relation['idx1'] = relation['ensembl1'].map(node_id_map)
        relation['idx2'] = relation['ensembl2'].map(node_id_map)

        # replace ensembl with idx
        cols = relation.columns.to_list()
        cols = [col for col in cols if col not in ['pathway_name', 'ensembl1', 'ensembl2','idx1','idx2']]
        cols = ['pathway_name', 'idx1', 'idx2'] + cols
        relation = relation.loc[:,cols]

        return gene_counts, relation, node_id_map

    def _get_masks(self, relation:pd.DataFrame):
        # initialize empty dict
        mask_nodes = {}

        # iterate through df grouped by mask_id
        for mask_id, group in relation.groupby('pathway_name'):
            nodes = pd.concat([group['idx1'], group['idx2']]).unique() # get unique nodes in idx1 & idx2
            mask_nodes[mask_id] = nodes.tolist() # append to dict

        # list masks
        masks = [j for _,j in mask_nodes.items()]

        return masks
    
    def _filter_counts(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, y_col:str, y_drop:dict, max_subset:int):
            # drop cols by class
            if (y_drop != None) & (type(y_drop) == dict):
                for key, value in y_drop.items():
                    metadata = metadata[~metadata[key].isin(value)]

            # downsample
            if (max_subset != None) & (type(max_subset) == int):

                # helper fxn
                def downsample(group):
                    if len(group) > max_subset:
                        return group.sample(n=max_subset)
                    return group
                
                # apply downsampling
                metadata_grouped = metadata.groupby(y_col)
                metadata_grouped = metadata_grouped.apply(downsample, include_groups=False).reset_index(drop=True)

                # reappend y_col, reorder
                metadata = pd.merge(metadata_grouped, metadata[['barcode', y_col]], on='barcode', how='left')
                metadata = metadata[['barcode','type','subtype']]

            # apply filter
            gene_counts = gene_counts[metadata['barcode']]

            return gene_counts, metadata
    
    def _get_Xy(self, gene_counts:pd.DataFrame, metadata:pd.DataFrame, y_col:str):
        # get counts
        X = gene_counts.T
        y_labels = metadata[y_col]

        # format 
        X = X.values.astype(np.float32)
        y = pd.get_dummies(y_labels).values.astype(np.float32)
        y_labels = y_labels.unique().tolist()

        # add 3rd dim, e.g. reshape to [num_samples, num_nodes, 1]
        X = np.expand_dims(X, axis=-1)

        # to tensor
        X = torch.tensor(X)
        y = torch.tensor(y)

        return X, y, y_labels

    def _get_dims(self, X:torch.Tensor, y:torch.Tensor, masks:list):
        num_samples, num_nodes, num_features = X.shape
        _, num_labels = y.shape
        num_masks = len(masks)

        return num_samples, num_nodes, num_features, num_labels, num_masks
    
    def __str__(self):

        out = ''
        width = 16 # print col width

        for name, variable in self.__dict__.items():
            # get variable shape
            if type(variable) == pd.core.frame.DataFrame:
                shape = variable.shape
            elif type(variable) == torch.Tensor or type(variable) == np.ndarray:
                shape = tuple([i for i in variable.shape])
            elif type(variable) == list:
                shape = len(variable)
            elif type(variable) == int:
                shape = variable
            else:
                shape = None

            # append shape if applicable
            if shape != None:
                try:
                    out += f'{name:<{width}} {str(shape):<{width}} {type(variable).__name__} ({variable.device.__str__()})\n'
                except:
                    out += f'{name:<{width}} {str(shape):<{width}} {type(variable).__name__}\n'
            else:
                out += f'{name:<{width}} {type(variable).__name__}\n'

        # return string
        return out



In [397]:
brca = Data(
    tcga_project='TCGA-BRCA',
    metadata_subtype_col='paper_BRCA_Subtype_PAM50',
    y_col = 'subtype',
    y_drop = {'subtype':['Normal', 'Metastatic']},
)

**** Data() ****
log0_method      str
gene_counts      (4383, 1184)     DataFrame
metadata         (1184, 3)        DataFrame
relation         (75939, 19)      DataFrame
node_id_map      dict
masks            305              list
X                (1184, 4383, 1)  Tensor (cpu)
y                (1184, 6)        Tensor (cpu)
y_labels         6                list
num_samples      1184             int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [390]:
print_metadata(tcga_project='TCGA-GBM')

barcode (391): ['TCGA-19-0955-01A-02R-A96U-41', 'TCGA-06-5856-01A-01R-A96S-41', 'TCGA-06-5856-01A-01R-1849-01', '...']

patient (293): ['TCGA-19-0955', 'TCGA-06-5856', 'TCGA-06-0152', '...']

sample (303): ['TCGA-19-0955-01A', 'TCGA-06-5856-01A', 'TCGA-06-0152-01A', '...']

shortLetterCode (3): ['TP', 'TR', 'NT']

definition (3): ['Primary solid Tumor', 'Recurrent Solid Tumor', 'Solid Tissue Normal']

sample_submitter_id (303): ['TCGA-19-0955-01A', 'TCGA-06-5856-01A', 'TCGA-06-0152-01A', '...']

sample_type_id (3): [1, 2, 11]

tumor_descriptor (3): ['Primary', 'Recurrence', 'Not Applicable']

sample_id (303): ['2fd6e759-2d50-4130-bbaf-4018929058f5', 'fc746957-2a65-44b9-9819-e857c390a1de', '1b8efa87-5057-48d9-be73-cfb1955a16bb', '...']

submitter_id (293): ['TCGA-19-0955', 'TCGA-06-5856', 'TCGA-06-0152', '...']

sample_type (3): ['Primary Tumor', 'Recurrent Tumor', 'Solid Tissue Normal']

specimen_type (2): ['Unknown', 'Solid Tissue']

state (1): ['released']

is_ffpe (1): [False]

tiss

In [396]:
gbm = Data(
    tcga_project='TCGA-GBM',
    metadata_subtype_col='paper_Original.Subtype',
    y_col='subtype',
    y_drop={'subtype':['Recurrent Tumor', 'Primary Tumor']}
)

**** Data() ****
log0_method      str
gene_counts      (4383, 352)      DataFrame
metadata         (352, 3)         DataFrame
relation         (75939, 19)      DataFrame
node_id_map      dict
masks            305              list
X                (352, 4383, 1)   Tensor (cpu)
y                (352, 6)         Tensor (cpu)
y_labels         6                list
num_samples      352              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---